In [ ]:
%run "Conformant Search.ipynb"
import random
import time
from collections import deque
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- CSS Styling ---
custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght=400;500;600;700&family=JetBrains+Mono&display=swap');

.app-container { background-color: #f9f9ff; font-family: 'Inter', sans-serif; }
.modern-card { background-color: #ffffff; border: 1px solid #c3c6d7; border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); box-sizing: border-box; }
.card-header { font-size: 16px; font-weight: 600; color: #141b2b; border-bottom: 1px solid #c3c6d7; padding-bottom: 10px; margin-bottom: 15px; display: flex; justify-content: space-between; }

.stat-box { background-color: #ffffff !important; border: 1px solid #c3c6d7 !important; border-radius: 12px !important; padding: 15px !important; box-sizing: border-box; }

.puzzle-input input[type="number"] {
    font-size: 18px !important; font-weight: 700 !important; text-align: center !important; color: #004ac6 !important;
    background-color: #e1e8fd !important; border: 1px solid rgba(0,74,198,0.2) !important; border-radius: 8px !important;
    height: 100% !important; box-sizing: border-box;
}

.btn-primary { 
    background-color: #004ac6 !important; color: white !important; border-radius: 999px !important; 
    font-weight: 600 !important; border: 1px solid #004ac6 !important; width: 95% !important; box-sizing: border-box !important;
}
.btn-primary:hover { background-color: #003ea8 !important; }

.btn-action { background-color: #e1e8fd !important; color: #38485d !important; border-radius: 8px !important; font-weight: 600 !important; border: 1px solid rgba(0,74,198,0.2) !important; font-size: 11px !important; padding: 4px 8px !important; }

.log-output { background-color: #f1f3ff !important; font-family: 'JetBrains Mono', monospace !important; border: none !important; }

.anim-board { display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; max-width: 200px; margin: 0 auto; background-color: #f1f3ff; padding: 12px; border-radius: 12px; }
.anim-tile { aspect-ratio: 1; background-color: #ffffff; border: 1px solid #c3c6d7; border-radius: 8px; display: flex; align-items: center; justify-content: center; font-size: 20px; font-weight: 700; color: #004ac6; box-shadow: 0 1px 3px rgba(0,0,0,0.05); transition: all 0.3s ease; }
.anim-tile-empty { aspect-ratio: 1; background-color: rgba(220, 226, 247, 0.4); border: 2px dashed #c3c6d7; border-radius: 8px; }
</style>
"""

display(HTML(custom_css))

header_html = widgets.HTML(value="""
<div style="display: flex; justify-content: space-between; align-items: center; padding: 15px 30px; background-color: #ffffff; border-bottom: 1px solid #c3c6d7; font-family: 'Inter', sans-serif;">
    <span style="font-size: 20px; font-weight: 700; color: #141b2b;">Conformant Search (BFS Early)</span>
    <div style="color: #004ac6; font-weight: 700; border-bottom: 2px solid #004ac6; padding-bottom: 4px; font-size: 14px;">BFS Early Goal Test</div>
</div>
""")

# Grids for Initial States
input_boxes_1 = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='45px')) 
                 for v in [1, 2, 3, 4, 0, 6, 7, 5, 8]]
for box in input_boxes_1: box.add_class('puzzle-input')
input_grid_1 = widgets.GridBox(input_boxes_1, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="5px"))

input_boxes_2 = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='45px')) 
                 for v in [1, 2, 3, 5, 0, 4, 6, 7, 8]]
for box in input_boxes_2: box.add_class('puzzle-input')
input_grid_2 = widgets.GridBox(input_boxes_2, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="5px"))

# Goal State Grid
goal_boxes = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='45px')) 
              for v in [1, 2, 3, 4, 5, 6, 7, 8, 0]]
for box in goal_boxes: box.add_class('puzzle-input')
goal_grid = widgets.GridBox(goal_boxes, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="5px"))

btn_random = widgets.Button(description="Random Start", layout=widgets.Layout(flex='1'))
btn_random.add_class('btn-action')

btn_reset = widgets.Button(description="Reset", layout=widgets.Layout(flex='1'))
btn_reset.add_class('btn-action')

btn_load_ex1 = widgets.Button(description="Load Bài Tập (Không giải được)", layout=widgets.Layout(flex='1'))
btn_load_ex1.add_class('btn-action')

btn_load_ex2 = widgets.Button(description="Load Ví dụ Giải được", layout=widgets.Layout(flex='1'))
btn_load_ex2.add_class('btn-action')

action_btns_1 = widgets.HBox([btn_random, btn_reset], layout=widgets.Layout(gap='5px', margin='10px 0 5px 0'))
action_btns_2 = widgets.HBox([btn_load_ex1, btn_load_ex2], layout=widgets.Layout(gap='5px'))

initial_state_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>Trạng Thái Ban Đầu (Initial Belief State)</span></div>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px; font-weight:600; color:#54647a; display:block; margin-bottom:5px;">Ma Trận 1</span>'), input_grid_1], layout=widgets.Layout(flex='1')),
        widgets.VBox([widgets.HTML('<span style="font-size:12px; font-weight:600; color:#54647a; display:block; margin-bottom:5px;">Ma Trận 2</span>'), input_grid_2], layout=widgets.Layout(flex='1'))
    ], layout=widgets.Layout(gap='15px')),
    action_btns_1, action_btns_2
], layout=widgets.Layout(margin='0 0 20px 0'))
initial_state_card.add_class('modern-card')

btn_bfs_search = widgets.Button(description="Giải Conformant (BFS Early)", layout=widgets.Layout(height='45px'))
btn_bfs_search.add_class('btn-primary')

config_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>Trạng Thái Đích & Điều Khiển</span></div>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px; font-weight:600; color:#54647a; display:block; margin-bottom:5px;">Ma Trận Đích (Goal State)</span>'), goal_grid], layout=widgets.Layout(flex='1')),
        widgets.VBox([
            widgets.HTML('<div style="font-size:11px; color:#54647a; font-style:italic; margin-bottom:10px;">BFS Early Goal Test sẽ tìm kiếm một chuỗi hành động duy nhất giải được cho cả hai ma trận xuất phát cùng lúc (Belief State).</div>'),
            btn_bfs_search
        ], layout=widgets.Layout(flex='1', justify_content='center', padding='0 0 0 10px'))
    ], layout=widgets.Layout(gap='15px'))
], layout=widgets.Layout(margin='0 0 20px 0'))
config_card.add_class('modern-card')

step_label = widgets.HTML('<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Ready</span>')
sim_header = widgets.HBox([
    widgets.HTML('<span style="font-size: 16px; font-weight: 600; color: #141b2b;">Mô Phỏng Trực Quan (Simulation)</span>'),
    step_label
], layout=widgets.Layout(justify_content='space-between', border_bottom='1px solid #c3c6d7', padding='0 0 10px 0', margin='0 0 15px 0', width='100%'))

anim_html = widgets.HTML(value="")
sim_card = widgets.VBox([sim_header, anim_html], layout=widgets.Layout(flex='1'))
sim_card.add_class('modern-card')

left_col = widgets.VBox([initial_state_card, config_card, sim_card], layout=widgets.Layout(flex='1.8', min_width='60%'))

stat_steps = widgets.HTML()
stat_nodes = widgets.HTML()
stat_time = widgets.HTML()
stat_status = widgets.HTML()

def update_stat(html_widget, value):
    html_widget.value = f'<div style="font-size: 18px; font-weight: 700; color: #141b2b; text-align: center; height: 30px; display: flex; align-items: center; justify-content: center;">{value}</div>'

update_stat(stat_steps, "-")
update_stat(stat_nodes, "-")
update_stat(stat_time, "-")
update_stat(stat_status, "-")

def make_stat_box(title, html_widget):
    box = widgets.VBox([
        widgets.HTML(f'<span style="font-size: 10px; font-weight: 600; color: #54647a; text-transform: uppercase; display: block; text-align: center; width: 100%;">{title}</span>'),
        html_widget
    ], layout=widgets.Layout(width='100%', align_items='center'))
    box.add_class('stat-box')
    return box

stat_grid = widgets.GridBox([
    make_stat_box("Steps", stat_steps),
    make_stat_box("Nodes Generated", stat_nodes),
    make_stat_box("Time", stat_time),
    make_stat_box("Status", stat_status)
], layout=widgets.Layout(grid_template_columns="1fr 1fr", gap="12px", margin="0 0 15px 0"))

log_content = widgets.HTML(value='')
out_text = widgets.VBox([log_content], layout=widgets.Layout(flex='1', overflow='auto', padding='10px', max_height='420px'))
out_text.add_class('log-output')

log_card = widgets.VBox([
    widgets.HTML('<div style="display:flex; align-items:center; justify-content:space-between; border-bottom: 1px solid #c3c6d7; padding: 10px 15px; background-color: #e1e8fd; border-radius: 12px 12px 0 0;"><span style="font-size: 12px; font-weight: 700; color: #141b2b; text-transform: uppercase; letter-spacing: 0.5px;">Lịch Sử Tìm Kiếm (Log)</span></div>'),
    out_text
], layout=widgets.Layout(background_color='#f1f3ff', border='1px solid #c3c6d7', border_radius='12px', flex='1'))

right_col = widgets.VBox([stat_grid, log_card], layout=widgets.Layout(flex='1', min_width='280px'))

main_app = widgets.VBox([
    header_html,
    widgets.HBox([left_col, right_col], layout=widgets.Layout(padding='15px', gap='15px'))
])
main_app.add_class('app-container')

def render_boards(states):
    html_content = '<div style="display: flex; gap: 15px; justify-content: center; flex-wrap: wrap;">'
    for idx, state in enumerate(states):
        html_content += f'<div style="text-align: center;"><div style="font-weight: 600; font-size: 11px; color: #54647a; margin-bottom: 5px;">Bản Đồ {idx+1}</div>'
        html_content += '<div class="anim-board">'
        for val in state:
            if val == 0:
                html_content += '<div class="anim-tile-empty"></div>'
            else:
                html_content += f'<div class="anim-tile">{val}</div>'
        html_content += '</div></div>'
    html_content += '</div>'
    anim_html.value = html_content


def print_log_belief_state(step_title, action, belief_state):
    if action:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">Bước {step_title}: Di chuyển ô trống sang <span style="color: #004ac6;">{action}</span></p>'
    else:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">{step_title}</p>'
    log_html = '<div style="margin-bottom: 15px;">' + action_html
    log_html += '<div style="display: flex; gap: 15px; flex-wrap: wrap;">'
    for idx, state in enumerate(belief_state):
        state_str = ""
        for row_idx in range(0, 9, 3):
            row = state[row_idx:row_idx+3]
            state_str += "  " + "    ".join([str(x) if x != 0 else "[ ]" for x in row]) + "\n"
        log_html += f'<div style="background-color: #ffffff; padding: 12px; border-radius: 8px; border: 1px solid rgba(195, 198, 215, 0.5); display: inline-block; min-width: 120px; text-align: center;">'
        log_html += f'<div style="font-size: 11px; font-weight: 600; color: #54647a; margin-bottom: 5px; border-bottom: 1px dashed rgba(195,198,215,0.5); padding-bottom: 3px;">Bản đồ {idx+1}</div>'
        log_html += '<pre style="margin: 0; font-family: monospace; font-size: 13px; line-height: 1.4; color: #141b2b; text-align: left;">' + state_str + '</pre>'
        log_html += '</div>'
    log_html += '</div></div><div style="border-top: 1px solid rgba(195, 198, 215, 0.5); margin-bottom: 15px; width: 100%;"></div>'
    log_content.value += log_html

def animate_path(start_states, path, goal_state):
    render_boards(start_states)
    step_label.value = f'<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Step 0/{len(path)}</span>'
    time.sleep(1.2)
    
    for step_idx, (action, next_belief) in enumerate(path):
        render_boards(list(next_belief))
        step_label.value = f'<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Step {step_idx + 1}/{len(path)}</span>'
        time.sleep(0.8)
        
    final_belief = path[-1][1] if path else frozenset(tuple(s) for s in start_states)
    if len(final_belief) == 1 and next(iter(final_belief)) == tuple(goal_state):
        step_label.value = '<span style="background:#d1f4e0; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#0d6e35;">Finished ✅</span>'
    else:
        step_label.value = '<span style="background:#fde8e8; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#c53030;">Failed ❌</span>'

def solve_and_animate():
    log_content.value = ''
    start_1 = [box.value for box in input_boxes_1]
    start_2 = [box.value for box in input_boxes_2]
    goal_state = [box.value for box in goal_boxes]
    start_states = [start_1, start_2]
    
    render_boards(start_states)
    log_content.value += f'<div style="color: #004ac6; font-weight: bold; margin-bottom: 15px; font-family: Inter; font-size: 13px;">Khởi chạy Conformant BFS Search...</div>'
    start_time = time.time()
    try:
        path, nodes_generated, log_data = conformant_bfs_solve(start_states, goal_state)
    except Exception as e:
        log_content.value += f'<div style="color: red;">❌ Lỗi thực thi: {str(e)}</div>'
        return
    end_time = time.time()
    elapsed_ms = int((end_time - start_time) * 1000)
    
    success = False
    if path is not None:
        final_belief = path[-1][1] if path else frozenset(tuple(s) for s in start_states)
        success = (len(final_belief) == 1 and next(iter(final_belief)) == tuple(goal_state))
        
    status_text = '<span style="color: #0d6e35; font-weight: bold;">SUCCESS</span>' if success else '<span style="color: #c53030; font-weight: bold;">FAILED</span>'
    
    update_stat(stat_steps, str(len(path)) if path else "-")
    update_stat(stat_nodes, f"{nodes_generated:,}")
    update_stat(stat_time, f"{elapsed_ms}ms")
    update_stat(stat_status, status_text)
    
    for log_entry in log_data:
        step_val = log_entry['step']
        if isinstance(step_val, str) and step_val == 'KQ':
            bg_color = '#fde8e8'
            border_color = 'rgba(197, 48, 48, 0.3)'
            step_color = '#c53030'
        else:
            bg_color = '#fcfcfc'
            border_color = 'rgba(0,74,198,0.1)'
            step_color = '#004ac6'
        log_content.value += f'<div style="background-color: {bg_color}; padding: 10px; border-radius: 8px; border: 1px solid {border_color}; margin-bottom: 10px; font-family: Inter; font-size: 12px; line-height: 1.4;">'
        log_content.value += f'<div style="font-weight: 700; color: {step_color}; border-bottom: 1px dashed {border_color}; padding-bottom: 4px; margin-bottom: 6px;">BƯỚC {step_val}</div>'
        log_content.value += f'<div>{log_entry["action_html"]}</div>'
        log_content.value += f'<div style="margin-top: 6px; font-size: 11px; color: #54647a; background-color: #f1f3ff; padding: 2px 6px; border-radius: 4px;"><b>{log_entry["frontier_str"]}</b> | <b>{log_entry["reached_str"]}</b></div>'
        log_content.value += '</div>'
        
    if path:
        log_content.value += '<div style="margin: 20px 0; border-top: 2px solid #004ac6; padding-top: 15px;"><b style="color: #004ac6;">MA TRẬN CÁC BƯỚC ĐI (BELIEF STATES):</b></div>'
        print_log_belief_state("Trạng thái bắt đầu", None, start_states)
        for step_idx, (act, next_belief) in enumerate(path):
            print_log_belief_state(str(step_idx + 1), act, list(next_belief))
        thread = threading.Thread(target=animate_path, args=(start_states, path, goal_state))
        thread.start()
    else:
        step_label.value = '<span style="background:#fde8e8; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#c53030;">No Solution</span>'

btn_bfs_search.on_click(lambda b: solve_and_animate())

def randomize_boards(b):
    nums1 = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    random.shuffle(nums1)
    for i, box in enumerate(input_boxes_1): box.value = nums1[i]
    
    nums2 = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    random.shuffle(nums2)
    for i, box in enumerate(input_boxes_2): box.value = nums2[i]
    
    render_boards([nums1, nums2])
btn_random.on_click(randomize_boards)

def reset_boards(b):
    nums = [0]*9
    for box in input_boxes_1: box.value = 0
    for box in input_boxes_2: box.value = 0
    render_boards([nums, nums])
btn_reset.on_click(reset_boards)

def load_example_1(b):
    # Bài tập từ người dùng (Không giải được do khác parity nghịch thế)
    nums1 = [1, 2, 3, 4, 0, 6, 7, 5, 8]
    nums2 = [1, 2, 3, 5, 0, 4, 6, 7, 8]
    goal = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    
    for i, box in enumerate(input_boxes_1): box.value = nums1[i]
    for i, box in enumerate(input_boxes_2): box.value = nums2[i]
    for i, box in enumerate(goal_boxes): box.value = goal[i]
    
    render_boards([nums1, nums2])
btn_load_ex1.on_click(load_example_1)

def load_example_2(b):
    # Ví dụ giải được (Plan: ['Xuống', 'Phải'])
    nums1 = [1, 2, 3, 4, 5, 0, 7, 8, 6]
    nums2 = [1, 2, 3, 4, 5, 6, 7, 0, 8]
    goal = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    
    for i, box in enumerate(input_boxes_1): box.value = nums1[i]
    for i, box in enumerate(input_boxes_2): box.value = nums2[i]
    for i, box in enumerate(goal_boxes): box.value = goal[i]
    
    render_boards([nums1, nums2])
btn_load_ex2.on_click(load_example_2)

# Khởi tạo mặc định với Bài Tập không giải được
load_example_1(None)
display(main_app)
